# Refit final `/recommend`

| | |
|---|---|
| **Objectif** | passer du modèle évalué (train 1991-2012, test 2013) au modèle servi en incorporant 2013 dans l'entraînement |
| **Entrées** | `data/processed/recommend_training_dataset.csv` ; coordonnées Natural Earth ; hyperparamètres figés du notebook 15 |
| **Sorties** | `models/recommend_model.joblib`, `models/recommend_model_metadata.json`, `models/recommend_context.json` |
| **Cible** | `yield_t_ha` ; **aucune nouvelle métrique de généralisation** — les métriques du modèle restent celles obtenues sur 2013 dans le notebook 15 |
| **Modèle** | ExtraTreesRegressor 150 arbres, mêmes hyperparamètres, même preprocessing, mêmes 9 features |

**Ce notebook 16 sert uniquement à produire l'artefact final destiné au serving, réentraîné avec toutes les données historiques disponibles. Le notebook 15 reste bien la référence méthodologique pour le choix et l'évaluation du modèle.**

# Pourquoi un refit final

- Le modèle a été **sélectionné** sans utiliser 2013 : validation temporelle 2008-2012 (notebook 14) puis choix figé d'ExtraTrees dans le notebook 15.
- Il a ensuite été **évalué une seule fois** sur 2013 (notebook 15) : `RMSE 1,6584 t/ha`, `MAE 0,7615`, `R² 0,9638`. Ces métriques restent la mesure indépendante de généralisation.
- Une fois cette évaluation figée, la ligne 2013 de chaque pays peut être incorporée dans l'artefact destiné au serving : elle a été gardée à part uniquement pour l'évaluation, pas pour un tuning supplémentaire.
- Le refit ne modifie **pas** le modèle (mêmes hyperparamètres), **pas** le preprocessing, **pas** les 9 features. Il applique le même pipeline aux 15 636 lignes admissibles jusqu'en 2013.

**Année cible technique.** L'artefact ainsi obtenu correspond à des données historiques jusqu'en 2013 ; la variable `year=2014` est la première année non couverte par l'historique et sert de convention interne pour préremplir les moyennes 3 ans (2011-2013). Le projet est présenté en 2026 et l'application ne prétend pas fournir de recommandations agricoles courantes.

# Reconstruction du dataset

Mêmes fonctions du package que le notebook 15 : `load_dataset`, `charger_coordonnees`, `add_recommend_features`, `add_historical_conditions`, `temporal_split`. Aucun feature engineering nouveau. Aucun changement de logique. Aucune fuite temporelle.

In [1]:
import json
import platform
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import ExtraTreesRegressor

from agritech.config import PATHS, SEED
from agritech.geo import charger_coordonnees
from agritech.modeling import experiment_pipeline
from agritech.preprocessing import PREPROCESSING_DESCRIPTION
from agritech.recommend_config import (
    RECOMMEND_COLUMNS,
    RECOMMEND_CROPS,
    RECOMMEND_DATASET,
    RECOMMEND_GEOGRAPHY,
    RECOMMEND_HISTORICAL_CONDITIONS,
    RECOMMEND_ROWS,
    RECOMMEND_TARGET,
    RECOMMEND_TEST_YEAR,
)
from agritech.recommend_features import add_historical_conditions, add_recommend_features
from agritech.recommend_service import preparer_demande
from agritech.training_data import load_dataset, temporal_split

In [2]:
df = load_dataset(
    RECOMMEND_DATASET, RECOMMEND_ROWS, RECOMMEND_COLUMNS,
    non_negative=[RECOMMEND_TARGET], verbose=False,
)
coord = charger_coordonnees()
df = add_historical_conditions(add_recommend_features(df, coord))

CATEGORIELLES = ["crop"]
NUMERIQUES = ["year", *RECOMMEND_HISTORICAL_CONDITIONS, *RECOMMEND_GEOGRAPHY]
VARIABLES = ["iso3", *CATEGORIELLES, *NUMERIQUES]

X_train, X_test, y_train, y_test = temporal_split(
    df, VARIABLES, RECOMMEND_TARGET, RECOMMEND_TEST_YEAR, verbose=False,
)
avec_historique_train = X_train["temp_hist"].notna().to_numpy()
avec_historique_test = X_test["temp_hist"].notna().to_numpy()
X_hist, y_hist = X_train[avec_historique_train], y_train[avec_historique_train]
X_test_hist, y_test_hist = X_test[avec_historique_test], y_test[avec_historique_test]

assert len(X_hist) == 14_941, f"X_hist attendu 14 941, obtenu {len(X_hist)} : le chemin de préparation a divergé"
assert len(X_test_hist) == len(X_test), "certaines lignes 2013 n'ont pas d'historique"

print(f"X_hist  train 1991-2012 : {len(X_hist)} lignes")
print(f"X_test  année 2013      : {len(X_test_hist)} lignes (100 % avec historique)")

X_hist  train 1991-2012 : 14941 lignes
X_test  année 2013      : 695 lignes (100 % avec historique)


# Dataset de refit 1991-2013

In [3]:
X_refit = pd.concat([X_hist, X_test_hist], ignore_index=True)
y_refit = pd.concat([y_hist, y_test_hist], ignore_index=True)

assert len(X_refit) == 15_636, f"X_refit attendu 15 636, obtenu {len(X_refit)}"

print(f"période refit         : 1991-2013")
print(f"nombre de lignes      : {len(X_refit)}")
print(f"nombre de pays        : {X_refit['iso3'].nunique()}")
print(f"nombre de cultures    : {X_refit['crop'].nunique()}")
print(f"features utilisées    : {CATEGORIELLES + NUMERIQUES}")

période refit         : 1991-2013
nombre de lignes      : 15636
nombre de pays        : 115
nombre de cultures    : 10
features utilisées    : ['crop', 'year', 'temp_hist', 'rain_mm', 'log_pest_hist', 'lat_abs', 'geo_x', 'geo_y', 'geo_z']


# Pipeline final figé

Mêmes hyperparamètres que ceux choisis dans le notebook 15 (cellule 48). Aucun tuning, aucune nouvelle comparaison, aucune nouvelle validation croisée après avoir vu 2013.

In [4]:
REGLAGES_FINAUX = {
    "n_estimators": 150,
    "max_features": 0.9,
    "min_samples_split": 3,
    "random_state": SEED,
    "n_jobs": 1,
}
pipeline_refit = experiment_pipeline(
    ExtraTreesRegressor(**REGLAGES_FINAUX), CATEGORIELLES, NUMERIQUES,
)
pipeline_refit

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, defa

# Refit final

`fit` sur les 15 636 lignes 1991-2013. **Aucune métrique n'est calculée sur ce dataset** : la performance finale reste celle mesurée sur 2013 dans le notebook 15.

In [5]:
pipeline_refit.fit(X_refit[CATEGORIELLES + NUMERIQUES], y_refit)
print(f"pipeline entraîné sur {len(X_refit)} lignes 1991-2013")

pipeline entraîné sur 15636 lignes 1991-2013


# Recalcul du training domain

Le `training_domain` décrit désormais les valeurs réellement présentées au modèle servi. Il est calculé sur `X_refit`. 2013 n'élargit que la borne haute de `log_pest_hist` : les bornes de `temp_hist` et `rain_mm` restent identiques à celles du notebook 15.

In [6]:
training_domain = {
    "temp_hist": {
        "min": float(X_refit["temp_hist"].min()),
        "max": float(X_refit["temp_hist"].max()),
        "unit": "°C",
    },
    "rain_mm": {
        "min": float(X_refit["rain_mm"].min()),
        "max": float(X_refit["rain_mm"].max()),
        "unit": "mm",
    },
    "log_pest_hist": {
        "min": float(X_refit["log_pest_hist"].min()),
        "max": float(X_refit["log_pest_hist"].max()),
        "unit": "log(t+1)",
    },
}
training_domain

{'temp_hist': {'min': 2.566666666666667,
  'max': 30.24666666666667,
  'unit': '°C'},
 'rain_mm': {'min': 51.0, 'max': 3240.0, 'unit': 'mm'},
 'log_pest_hist': {'min': 0.039220713153281295,
  'max': 14.39418266302968,
  'unit': 'log(t+1)'}}

# Metadata du modèle refitté

Structure séparée :

- au top-level : les clés requises par `agritech.serving.load_bundle` (`service`, `features`, `training_domain`, `model_version`, `created_on`) plus les descripteurs habituels du modèle ;
- `refit` : ce que l'artefact servi a effectivement vu ;
- `evaluation` : préserve exactement l'évaluation honnête du notebook 15 (métriques recopiées bit-identique).

In [7]:
model_path = PATHS.root / "models" / "recommend_model.joblib"
metadata_path = PATHS.root / "models" / "recommend_model_metadata.json"

# On ne redéfinit pas l'évaluation 2013 : on lit celle déjà écrite (notebook 15 puis lot API),
# on la contrôle contre les valeurs figées, puis on la conserve telle quelle dans le nouveau metadata.
existing_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
if "evaluation" in existing_metadata and "test_metrics" in existing_metadata["evaluation"]:
    saved_test_metrics = existing_metadata["evaluation"]["test_metrics"]  # rerun du notebook 16
else:
    saved_test_metrics = existing_metadata["final_test_metrics"]  # premier run après le notebook 15

assert saved_test_metrics["rmse"] == 1.6583653778656462, "métriques 2013 RMSE inattendues dans le metadata existant"
assert saved_test_metrics["mae"] == 0.7614744124700236, "métriques 2013 MAE inattendues dans le metadata existant"
assert saved_test_metrics["r2"] == 0.9638329168740789, "métriques 2013 R² inattendues dans le metadata existant"

encodeur = pipeline_refit["preprocessing"].named_transformers_["categorical"]
categorical_values = {
    nom: modalites.tolist()
    for nom, modalites in zip(CATEGORIELLES, encodeur.categories_)
}

evaluation = {
    "period_train": "1991-2012",
    "period_test": RECOMMEND_TEST_YEAR,
    "n_train": len(X_hist),
    "n_test": len(X_test_hist),
    "cv_strategy": "expanding_window_by_year",
    "cv_folds": 5,
    "cv_validation_years": "2008-2012",
    "test_metrics": saved_test_metrics,
    "notebook": "notebooks/15_recommend_final_model.ipynb",
}

refit = {
    "period": "1991-2013",
    "n_samples": len(X_refit),
    "trained_on": "all_rows_with_history_1991_to_2013",
    "target_year_for_serving": 2014,
}

metadata = {
    "service": "recommend",
    "model_type": type(pipeline_refit["model"]).__name__,
    "artifact": model_path.name,
    "target": RECOMMEND_TARGET,
    "features": CATEGORIELLES + NUMERIQUES,
    "categorical_features": CATEGORIELLES,
    "numeric_features": NUMERIQUES,
    "categorical_values": categorical_values,
    "preprocessing": PREPROCESSING_DESCRIPTION,
    "model_params": {cle: pipeline_refit["model"].get_params()[cle] for cle in REGLAGES_FINAUX},
    "training_dataset": RECOMMEND_DATASET.name,
    "refit": refit,
    "evaluation": evaluation,
    "notebook": "notebooks/16_recommend_final_refit.ipynb",
    "created_on": date.today().isoformat(),
    "versions": {
        "python": platform.python_version(),
        "scikit-learn": sklearn.__version__,
        "numpy": np.__version__,
        "joblib": joblib.__version__,
    },
    "model_version": "2.0.0",
    "training_domain": training_domain,
}
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(f"metadata écrites : {metadata_path.relative_to(PATHS.root)}")

metadata écrites : models/recommend_model_metadata.json


# Sauvegarde du modèle refitté

Compression LZMA niveau 3, comme dans le notebook 15.

In [8]:
joblib.dump(pipeline_refit, model_path, compress=("lzma", 3))
taille_mo = model_path.stat().st_size / 1e6
print(f"modèle sauvegardé : {model_path.relative_to(PATHS.root)} ({taille_mo:.1f} Mo)")

modèle sauvegardé : models/recommend_model.joblib (46.9 Mo)


# Données nécessaires pour produire une recommandation

Le modèle entraîné ne suffit pas à lui seul pour utiliser `/recommend`. Pour prédire les rendements des différentes cultures d'un pays, il attend notamment des variables historiques (`temp_hist`, `log_pest_hist`, `rain_mm`) et géographiques (`lat_abs`, `geo_x`, `geo_y`, `geo_z`).

Ces informations sont donc préparées une fois lors de la génération des artefacts et enregistrées dans `models/recommend_context.json`. L'API pourra ainsi construire les 10 candidats à comparer sans avoir à recharger ni retraiter les datasets d'entraînement.

Pour chacun des 115 pays, ce fichier contient :

- le nom du pays et sa pluviométrie ;
- les variables géographiques utilisées par le modèle ;
- les températures et quantités de pesticides des années 2011 à 2013, nécessaires au calcul des moyennes historiques sur 3 ans ;
- les cultures observées dans les données.

L'année 2014 est utilisée comme première année suivant l'historique disponible : les variables historiques associées sont donc calculées à partir de 2011-2013. Il s'agit d'une convention technique liée au dataset, et non de l'année actuelle d'utilisation de l'application.

In [9]:
HISTORY_YEARS = [2011, 2012, 2013]
TARGET_YEAR = 2014

geo_par_iso3 = (
    df.drop_duplicates("iso3").set_index("iso3")[["lat_abs", "geo_x", "geo_y", "geo_z"]]
)

countries = {}
for iso3 in sorted(df["iso3"].unique()):
    sub = df[df["iso3"] == iso3]
    hist = (
        sub[sub["year"].isin(HISTORY_YEARS)]
        .drop_duplicates(["iso3", "year"])
        .sort_values("year")
    )
    countries[iso3] = {
        "country": str(sub["area"].iloc[0]),
        "rain_mm": float(sub["rain_mm"].iloc[0]),
        "geography": {
            "lat_abs": float(geo_par_iso3.loc[iso3, "lat_abs"]),
            "geo_x":   float(geo_par_iso3.loc[iso3, "geo_x"]),
            "geo_y":   float(geo_par_iso3.loc[iso3, "geo_y"]),
            "geo_z":   float(geo_par_iso3.loc[iso3, "geo_z"]),
        },
        "history_2011_2013": [
            {"year": int(row.year), "avg_temp": float(row.avg_temp), "pesticides_t": float(row.pesticides_t)}
            for row in hist.itertuples()
        ],
        "observed_crops": sorted(sub["crop"].unique().tolist()),
    }

# Invariants du contexte
assert len(countries) == 115, f"115 pays attendus, {len(countries)} obtenus"
for iso3, entry in countries.items():
    years = [item["year"] for item in entry["history_2011_2013"]]
    assert years == HISTORY_YEARS, f"{iso3} : historique attendu {HISTORY_YEARS}, obtenu {years}"
    assert entry["observed_crops"], f"{iso3} : aucune culture observée"
union_crops = {c for entry in countries.values() for c in entry["observed_crops"]}
assert union_crops == set(RECOMMEND_CROPS), (
    f"union des cultures ≠ 10 attendues : manquantes {sorted(set(RECOMMEND_CROPS) - union_crops)}"
)

context_path = PATHS.root / "models" / "recommend_context.json"
context_payload = {
    "generated_from": {
        "dataset": RECOMMEND_DATASET.name,
        "geojson": "ne_110m_admin_0_countries.geojson",
    },
    "history_years": HISTORY_YEARS,
    "target_year": TARGET_YEAR,
    "countries": countries,
}
context_path.write_text(
    json.dumps(context_payload, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(f"contexte écrit : {context_path.relative_to(PATHS.root)} ({context_path.stat().st_size/1024:.1f} Ko)")
print(f"pays           : {len(countries)}")
print(f"cultures       : {sorted(union_crops)}")

contexte écrit : models/recommend_context.json (86.1 Ko)
pays           : 115
cultures       : ['Cassava', 'Maize', 'Plantains and others', 'Potatoes', 'Rice, paddy', 'Sorghum', 'Soybeans', 'Sweet potatoes', 'Wheat', 'Yams']


# Contrôles finaux

Relecture depuis le disque, vérification des invariants clés, croisement direct sur les 10 candidats 2014 de la France.

In [10]:
# 1. Rechargement du pipeline sauvegardé
modele_recharge = joblib.load(model_path)
assert list(modele_recharge.feature_names_in_) == CATEGORIELLES + NUMERIQUES
assert type(modele_recharge["model"]).__name__ == "ExtraTreesRegressor"
assert {k: modele_recharge["model"].get_params()[k] for k in REGLAGES_FINAUX} == REGLAGES_FINAUX

# 2. Rechargement des metadata et cohérence
meta_relue = json.loads(metadata_path.read_text(encoding="utf-8"))
assert meta_relue["model_version"] == "2.0.0"
assert meta_relue["refit"]["n_samples"] == 15_636
assert meta_relue["refit"]["period"] == "1991-2013"
assert meta_relue["evaluation"]["test_metrics"] == saved_test_metrics, "métriques 2013 relues != métriques 2013 récupérées avant écriture"
assert set(meta_relue["training_domain"]) == {"temp_hist", "rain_mm", "log_pest_hist"}

# 3. Rechargement du contexte
ctx_relu = json.loads(context_path.read_text(encoding="utf-8"))
assert len(ctx_relu["countries"]) == 115
assert ctx_relu["target_year"] == TARGET_YEAR
assert ctx_relu["history_years"] == HISTORY_YEARS

# 4. Croisement : construire les 10 candidats 2014 pour FRA via preparer_demande, puis scorer.
#
# Rappel technique : 2014 est ici la PREMIÈRE année suivant la fin du dataset historique (2013).
# C'est une convention interne permettant à `preparer_demande` d'utiliser l'historique 2011-2013
# pour construire les features. Ce n'est pas l'année d'utilisation de l'application : le projet
# est présenté en 2026 et l'application ne doit pas être présentée comme « pour 2014 ».
historique_pays = (
    df.loc[df["year"] < 2014, ["iso3", "year", "avg_temp", "rain_mm", "pesticides_t"]]
    .drop_duplicates(["iso3", "year"])
)
demande_fra = preparer_demande(historique_pays, "FRA", 2014, coord)
predictions_fra = modele_recharge.predict(demande_fra[CATEGORIELLES + NUMERIQUES])
assert predictions_fra.shape == (10,), f"10 prédictions attendues, obtenues {predictions_fra.shape}"
print(f"prédictions 2014 FRA (10 cultures) : min={predictions_fra.min():.2f} max={predictions_fra.max():.2f} t/ha")

print(f"\ntaille modèle joblib : {model_path.stat().st_size / 1e6:.1f} Mo")

prédictions 2014 FRA (10 cultures) : min=2.65 max=42.24 t/ha

taille modèle joblib : 46.9 Mo


# Conclusion

- Modèle refitté sur 15 636 lignes 1991-2013 (14 941 + 695), mêmes hyperparamètres, même preprocessing.
- Metadata mises à jour : `model_version = "2.0.0"`, section `refit` distincte de la section `evaluation` qui préserve exactement les métriques 2013 du notebook 15.
- Contexte de serving généré depuis les mêmes objets que le refit : 115 pays, historique 2011-2013 complet, 10 cultures observées au total.
- Trois artefacts prêts pour `/recommend` : `recommend_model.joblib`, `recommend_model_metadata.json`, `recommend_context.json`.
- Le notebook 16 est désormais l'unique source reproductible de ces artefacts ; le script `scripts/build_recommend_context.py` a été supprimé pour éviter deux sources de vérité.